# Curva de Phillips de Chile
## Inflación, desocupación y remuneraciones reales

La muestra común se calcula con la última publicación disponible de cada fuente oficial. El IPC empalmado tiene niveles desde diciembre de 2009, pero sus tasas anuales publicadas y el IR anual comienzan en enero de 2011.

Al ejecutar todas las celdas, se descargan y validan las cuatro fuentes oficiales antes de calcular el período común. Requiere internet. Para reproducir el corte local, cambia ACTUALIZAR_DATOS a False en la primera celda.

**Primera vez:** abre `lanzadores/02_abrir_jupyter.bat`, espera a que aparezca el navegador y selecciona **Run → Run All Cells** (Ejecutar todas las celdas). También puedes ejecutar cada celda con **Shift + Enter**. El asterisco `[*]` significa que Python está trabajando.

El gráfico al final incluye **Play**, pausa, reinicio y un selector mensual. Cada fotograma mantiene el rastro de los meses anteriores. El borde distingue el año fuera de pandemia; durante pandemia conserva el gris segmentado. El panel inferior incluye una leyenda para IMACEC e IPC. La versión HTML guardada en `resultados` funciona sin Jupyter y sin conexión.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'codigo'/'phillips.py').is_file())
sys.path.insert(0, str(ROOT/'codigo'))
import pandas as pd
from IPython.display import display, HTML
import importlib
import os
from datetime import datetime
if os.environ.get('PHILLIPS_SIN_ACTUALIZAR') != '1':
    os.environ['PHILLIPS_FECHA'] = datetime.now().strftime('%Y%m%d_%H_%M')
# Cambia a 'original' o 'sa' y vuelve a ejecutar todas las celdas.
IPC_AJUSTE = 'original'
os.environ['PHILLIPS_IPC'] = IPC_AJUSTE
import salidas, graficos
importlib.reload(salidas)
importlib.reload(graficos)
# El orquestador descarga una vez; al abrir el cuaderno manualmente se actualiza.
ACTUALIZAR_DATOS = os.environ.get('PHILLIPS_SIN_ACTUALIZAR') != '1'
if ACTUALIZAR_DATOS:
    from actualizar_datos import actualizar
    actualizar()
else:
    print('Corte local: no se descargaron fuentes en esta celda')
import phillips
importlib.reload(phillips)
from phillips import load_data, build_figure, export_results, static_chart, economic_summary, figure_html
print('Python:', sys.version.split()[0])
print('Proyecto:', Path.cwd().name)


## Qué representa cada variable

- **Eje horizontal:** tasa de desocupación nacional de la ENE, en porcentaje, sin ajuste estacional. Es un trimestre móvil, no una estimación mensual independiente.
- **Eje vertical:** variación a doce meses del **IPC General**, publicada en el CSV en modo `original` o calculada sobre el índice ajustado X-13/SEATS en modo `sa`. No se promedian divisiones del IPC ni se calcula un cambio a doce meses a partir de índices con bases distintas.
- **Área de la burbuja:** proporcional al valor absoluto de la variación anual del índice de remuneraciones **real**, por hora. El signo del IR se consulta al pasar el cursor; el color representa el IMACEC, no el salario real. Un centro oscuro permite localizar una variación exactamente cero sin asignarle un área económica ficticia.

El IR ya está deflactado por IPC; restar nuevamente la inflación sería un error. Se usa `var_12`, contrastada con los niveles separados por doce meses.

**Fechas:** seguimos la convención de los CSV: ENE se asigna al mes central (por ejemplo, mayo–julio corresponde a junio). Esto sirve para una comparación retrospectiva; el punto no representa información disponible en tiempo real en junio. La fecha final del trimestre también queda en la tabla.


In [ ]:
datos, cobertura, excluidos = load_data(ipc_ajuste=IPC_AJUSTE)
display(cobertura)
print(f'Período común: {datos.mes.iloc[0]} a {datos.mes.iloc[-1]} · {len(datos)} observaciones')
print(f'Meses excluidos por falta de alguna variable: {len(excluidos)}')
display(datos[['mes','Trimestre','fecha_final_ene','desocupacion','ipc_anual','ir_real_anual','imacec_promedio_anual','imacec_sa_anual','imacec_acumulado_anual','estado_ir']].tail(12).round(3))


## Elegir IPC original o desestacionalizado

En la primera celda cambia `IPC_AJUSTE` a `original` o `sa` y ejecuta todas las celdas. El modo SA requiere el ejecutable oficial: `python scripts/00_instalar_x13.py`. X-13 ajusta el nivel empalmado completo y calcula después tasas mensuales y de 12 meses. Exporta `ipc_comparacion.csv`, especificación y diagnóstico. Es experimental, revisable y no oficial del INE; no ajusta ENE ni IR. El indicador de diciembre usa el IPC seleccionado. Los archivos SA agregan `ipc_sa` para no reemplazar los originales.


## Comprobaciones antes de graficar

El código exige fechas únicas, uniones uno a uno, valores finitos y continuidad mensual. Nunca reemplaza faltantes por cero. La tabla `20261023_07_12_meses_excluidos.csv` permite auditar la intersección de las series. El archivo `20261023_07_12_fuentes_sha256.json` identifica la versión exacta de los CSV utilizados.


In [ ]:
assert not datos.fecha.duplicated().any()
assert datos[['desocupacion','ipc_anual','ir_real_anual','imacec_promedio_anual','imacec_sa']].notna().all().all()
assert datos.desocupacion.between(0,100).all()
print('Validaciones de las cuatro variables: correctas')
display(datos[['desocupacion','ipc_anual','ir_real_anual','imacec_promedio_anual','imacec_sa']].describe().round(3))


## Referencias de inflación y desempleo

La línea horizontal señala la **meta de inflación del 3%** del BCCh, definida para un horizonte de dos años; no exige que cada dato mensual sea 3%.

La banda vertical representa el rango **8,0–8,5% para 2024-T3** publicado en la minuta *Holguras en el mercado laboral*, citada en el IPoM de diciembre de 2024 (páginas 34 y 38 del PDF). Reúne estimaciones mediante filtros de Kalman multivariados y modelos VAR. La línea **8,25%** es el punto medio calculado por este proyecto, **no una estimación puntual oficial ni una cifra del IPoM de junio de 2026**. La banda no es un intervalo de confianza.

Es una referencia histórica fija, no una trayectoria estimada para 2011–2026. Además, la referencia utiliza desempleo desestacionalizado y nuestros puntos usan ENE sin ajuste estacional: las distancias son ilustrativas, no brechas cíclicas oficiales. NAIRU y tasa natural de largo plazo no son conceptos necesariamente idénticos.

Fuentes: [minuta BCCh diciembre de 2024](https://www.bcentral.cl/documents/33528/6735463/Minutas%2Bcitadas%2Ben%2Bel%2BIPoM%2Bdiciembre%2B2024.pdf/d24985ae-cb5e-2f03-4499-ecfad3d86ade) y [IPoM junio de 2026, meta de inflación, página 3](https://www.bcentral.cl/documents/33528/8413153/IPoM%2Bjunio%2B2026.pdf/93388589-0929-4ad6-b166-10981ca34946). Parámetros y trazabilidad: `referencias_macro.json` y `fuentes/REFERENCIAS.md`.


## IMACEC y fórmulas del gráfico

El promedio es de **tres meses**, centrado en el mes de la ENE. Se utiliza la serie **original** del IMACEC para el color y la serie **desestacionalizada oficial** para el panel inferior. Ambas son índices empalmados, promedio 2018=100. No se vuelve a desestacionalizar ni se confunden niveles con tasas.

Sea $I_t$ el índice original y $S_t$ el índice desestacionalizado mensual:

$$\bar I_t=(I_{t-1}+I_t+I_{t+1})/3$$

$$g_t=100\left(\frac{\bar I_t}{\bar I_{t-12}}-1\right)$$

El color usa $g_t$: **variación interanual del promedio de niveles**, no promedio de tres tasas interanuales. Por ejemplo, junio de 2026 compara mayo–julio de 2026 con mayo–julio de 2025. Requiere el dato del mes siguiente: es una visualización retrospectiva, no una señal disponible en tiempo real en junio. Con la alineación alternativa al mes final se usa $(I_{t-2}+I_{t-1}+I_t)/3$.

El panel inferior muestra las tasas en 12 meses del IMACEC desestacionalizado y del IPC:

$$a_t=100(S_t/S_{t-12}-1),\qquad \pi_t=100(IPC_t/IPC_{t-12}-1).$$

La inflación se toma de la tasa oficial publicada. La variación mensual del IMACEC se conserva en los datos como $m_t=100(S_t/S_{t-1}-1)$, pero no es la serie del panel. El cursor y la línea oscura siguen el mes seleccionado; el resto es contexto retrospectivo.

En cada diciembre, un rombo marca el crecimiento del **promedio anual del IMACEC original**:

$$G_Y=100\left(\frac{\sum_{m=1}^{12} I_{Y,m}}{\sum_{m=1}^{12} I_{Y-1,m}}-1\right).$$

Un cuadrado marca la inflación **diciembre contra diciembre**. Coincide con la tasa IPC a 12 meses publicada en diciembre; no se suman tasas redondeadas. No se dibujan cierres de años incompletos.

Tamaño y color de los puntos de diciembre usan una normalización común a las **tasas mensuales en 12 meses**, sobre toda la muestra:

$$L=\max(0.1,\max_t|a_t|,\max_t|\pi_t|),\quad n_t=v_t/L,\quad A_t=k|n_t|.$$

El color usa naranja en $-1$, claro en $0$ y azul en $+1$. La forma identifica la serie. La coordenada vertical muestra el acumulado, mientras tamaño y color codifican la tasa a 12 meses: ambas se detallan en el tooltip. La escala permanece fija en todos los fotogramas.

Cada enero de Phillips lleva una etiqueta MM-AAAA, además de 03-2020 y 08-2023. Los círculos entre marzo de 2020 y agosto de 2023, inclusive, tienen borde segmentado: período de pandemia Covid-19 en Chile **definido para esta visualización**. No se identifica este intervalo con toda la vigencia jurídica de la alerta sanitaria. Véase [Gobierno de Chile, fin de alerta el 31 de agosto de 2023](https://www.gob.cl/noticias/fin-alerta-sanitaria-covid-19-coronavirus-enfermedades-respiratorias-mascarillas-teletrabajo/).

**Otras variables y referencias**

- Desocupación: $u_t=100D_t/F_t$, donde $D$ son personas desocupadas y $F$ la fuerza de trabajo del trimestre móvil. Se usa la tasa publicada por INE.
- Inflación anual: $\pi_t=100(IPC_t/IPC_{t-12}-1)$. Se utiliza la variación publicada del IPC General, sin dividir índices de bases distintas.
- IR real anual: $r_t=100(IR^R_t/IR^R_{t-12}-1)$; se toma `var_12`, ya deflactada. Área de burbuja $A_t=k|r_t|$, con $k$ fijo; el signo del IR se consulta en el tooltip, no en el color. Si $r_t=0$, un centro fijo permite ubicar el punto.
- Normalización del color: $L=\max(\max_t|g_t|,0.1)$; $z_t=(g_t+L)/(2L)$. Naranja en $-L$, tono claro en $0$ y azul en $+L$. Escala simétrica y fija en todos los fotogramas, con barra lateral en porcentajes.
- Referencia vertical $u^*=(8.0+8.5)/2=8.25\%$, punto medio propio del rango BCCh histórico, no estimación puntual oficial. Distancias ilustrativas: $u_t-u^*$ y $\pi_t-3$, en puntos porcentuales.

La correlación descriptiva usa Pearson sobre las fechas comunes, sin interpretación causal:

$$\rho_{u,\pi}=\frac{\sum_t(u_t-\bar u)(\pi_t-\bar\pi)}{\sqrt{\sum_t(u_t-\bar u)^2\sum_t(\pi_t-\bar\pi)^2}}$$

**Precisión y reproducibilidad.** La tabla web BDE publica índices con un decimal. Las tasas derivadas son aproximaciones calculadas con esos índices redondeados; no se presentan como tasas oficiales exactas. Las series pueden revisarse. El CSV local permite ejecutar el cuaderno sin conexión. La descarga y sus códigos, fecha y hashes están en `actualizar_imacec.py` y `fuentes/imacec_metadata.json`. El cuaderno descarga y valida las cuatro fuentes por defecto; la reproducción sin descarga debe seleccionarse explícitamente.

Series BCCh: `F032.IMC.IND.Z.Z.EP18.Z.Z.0.M` (original) y `F032.IMC.IND.Z.Z.EP18.Z.Z.1.M` (desestacionalizada). [Descripción oficial del IMACEC](https://www.bcentral.cl/areas/estadisticas/imacec).

La ejecución `python scripts/05_ejecutar_todo.py` actualiza y valida juntas las cuatro fuentes oficiales antes de generar el análisis. `python scripts/01_actualizar_datos.py` actualiza solo los insumos. Una descarga o validación fallida detiene el proceso y conserva el corte anterior, sin analizarlo como si fuera nuevo. `--sin-actualizar-datos` permite reproducir explícitamente el corte local. La intersección, las correlaciones y el informe se recalculan según las fechas disponibles. La descarga UTC y los hashes se registran en `fuentes/actualizacion.json`.

El borde distingue el año fuera de la ventana de pandemia; dentro se conserva el borde gris segmentado. El relleno sigue codificando IMACEC. Las salidas comparten la fecha y hora local de inicio AAAAMMDD_HH_MM.

## Desestacionalización experimental del IPC

`ipc_x13.py` invoca el ejecutable oficial X-13ARIMA-SEATS y solicita explícitamente la descomposición SEATS (no X-11). Entrada: niveles mensuales positivos del IPC General empalmado, con fechas únicas y sin huecos. No se rellenan faltantes ni se concatenan índices de bases diferentes. Se usa toda la serie mensual disponible en la descarga oficial, no solo la intersección con ENE e IR.

Especificación: logaritmos, `automdl`, atípicos automáticos AO/LS/TC, horizonte interno de 36 meses y SEATS sin aproximación automática de modelos inadmisibles. No se añaden feriados extranjeros ni regresores de días hábiles. Los pronósticos internos sirven al filtro y no se agregan como observaciones a los gráficos. La serie final es `s11`, que conserva el irregular: no debe confundirse con la tendencia `s12` ni con una inflación subyacente. `s10` y `s13` se conservan para auditoría.

Para índice ajustado A_t: mensual = 100(A_t/A_{t-1}−1); anual = 100(A_t/A_{t-12}−1). El cierre de diciembre usa la tasa anual de la variante seleccionada. En modo original se conservan tasas oficiales y faltantes; no se reemplazan por cálculos con niveles redondeados. La tabla comparativa agrega el cálculo anual sobre niveles originales como control de redondeo.

La estimación no es oficial del INE. Se vuelve a estimar al cambiar los datos, especificación o ejecutable. Utiliza información de toda la muestra y sus extremos están sujetos a revisión; no es una estimación en tiempo real. Los diagnósticos se guardan sin ocultar advertencias ni sustituir fallos por otro método. La ejecución actual detecta un pico de días de negociación en residuos; debe evaluarse calendario chileno y estabilidad antes de un uso oficial. Desestacionalizar el IPC no transforma la ENE o el IR y no elimina los problemas causales de una curva de Phillips descriptiva.

Fuente del software y documentación: [U.S. Census Bureau](https://www.census.gov/data/software/x13as.X-13ARIMA-SEATS.html). [Manual de referencia](https://www2.census.gov/software/x-13arima-seats/x13as/unix-linux/documentation/docx13as.pdf).


## Comparación de la evolución por etapas

Se usan ventanas descriptivas coherentes con el gráfico: antes de marzo de 2020; marzo de 2020 a agosto de 2023; y desde septiembre de 2023 hasta el último mes común. Este corte no equivale a una declaración de emergencia sanitaria ni identifica efectos causales.

Para cada etapa se exportan observaciones, correlación IPC–desocupación, medias, extremos y meses correspondientes, además de los valores iniciales y finales. Los cambios de tasas se expresan en puntos porcentuales. Los cambios de niveles del IMACEC desestacionalizado y del índice de remuneraciones reales se calculan como 100 × (índice final / índice inicial − 1); no se suman las tasas anuales. El último mes también se compara con febrero de 2020.

Las medias de tasas interanuales no son tasas acumuladas ni crecimiento compuesto; las ventanas tienen longitudes distintas. La ENE usa trimestres móviles y se alinea al mes central. Los informes no interpretan las correlaciones, las diferencias de medias ni las referencias históricas como estimaciones causales. La tabla `evolucion_etapas.csv` y el resumen JSON conservan los cálculos utilizados por el Word, el artículo HTML y el cuaderno.


## Gráfico animado

Pulsa **Play** para recorrer la muestra y **Pausa** para detenerla. Puedes arrastrar el selector temporal; **Reiniciar** vuelve al primer mes. Pasa el cursor por una burbuja para ver las cifras y el trimestre de empleo.

Los ejes, la escala de áreas y la escala divergente del color permanecen fijos durante toda la animación. El panel inferior compara IMACEC desestacionalizado e IPC en variación a 12 meses, con cierres anuales en diciembre. Su cursor está sincronizado con el mes activo. Cada enero lleva fecha y la ventana Covid tiene bordes segmentados. Al volver a una fecha anterior, el rastro también retrocede.


In [ ]:
figura = build_figure(datos)
export_results(datos, cobertura, excluidos, figura)
static_chart(datos)
# HTML embebido: evita depender de extensiones de Plotly o de una conexión a internet.
display(HTML(figure_html(figura, full_html=False)))


## Lectura económica y sensibilidad de fechas

Una nube de puntos es una descripción, no una estimación causal de la curva de Phillips. La inflación depende también de expectativas, oferta, precios externos, tipo de cambio y política monetaria. La variación del IR real comparte el IPC como deflactor, por lo que no constituye una variable independiente de la inflación.

La comparación por subperíodos muestra cómo cambia la asociación según la ventana elegida. La ampliación incorpora caídas y rebotes de actividad y episodios de alta inflación. Las burbujas claras pueden representar variaciones pequeñas respecto de los extremos históricos, no necesariamente cero.

La correlación siguiente usa niveles contemporáneos, sin controles ni correcciones por autocorrelación. Los trimestres móviles se superponen y las tasas anuales también comparten meses. Por eso no se presentan pruebas de significancia ni recomendaciones de tasas de interés. Tampoco se estima una NAIRU propia; se incorpora únicamente la referencia histórica externa descrita arriba.

Para explorar la convención temporal, también se calcula la correlación con ENE asignada al mes final, restringiendo ambas alternativas a las mismas fechas. Este contraste cambia el emparejamiento, no el dato original.


In [ ]:
alternativa, _, _ = load_data(alignment='final', ipc_ajuste=IPC_AJUSTE)
fechas_comunes = datos.fecha[datos.fecha.isin(alternativa.fecha)]
a = datos[datos.fecha.isin(fechas_comunes)]
b = alternativa[alternativa.fecha.isin(fechas_comunes)]
resumen = pd.DataFrame([
    {'alineación': 'Mes central', 'n': len(a), 'correlación IPC-desocupación': a.desocupacion.corr(a.ipc_anual)},
    {'alineación': 'Mes final', 'n': len(b), 'correlación IPC-desocupación': b.desocupacion.corr(b.ipc_anual)}
])
display(resumen.round(3))
estadisticas = economic_summary(datos)
print(f"Correlación muestra completa: {estadisticas['correlacion']:.3f}")
display(pd.DataFrame(estadisticas['subperiodos']).round(3))
print('Subperíodos descriptivos predefinidos; no son regímenes estimados ni pruebas causales.')

print(estadisticas['evolucion']['regla'])
etapas = estadisticas['evolucion']['etapas']
display(pd.DataFrame([dict(etapa=e['etapa'], desde=e['desde'], hasta=e['hasta'], meses=e['n'],
    correlacion=e['correlacion'], **{k+'_media':v['media'] for k,v in e['metricas'].items()})
    for e in etapas]).round(3))
from analisis_evolucion import secciones_evolucion
for tipo, texto in secciones_evolucion(estadisticas):
    print()
    print(texto)

display(datos[['mes','distancia_meta_ipc_pp','distancia_referencia_nairu_pp']].tail(1).round(3))
display(datos.groupby(datos.fecha.dt.year)[['desocupacion','ipc_anual','ir_real_anual','imacec_promedio_anual','imacec_sa']].mean().round(3))
print('Advertencia: el último año puede ser parcial; no comparar promedios como años completos.')


## Archivos de salida y actualización

- `resultados/20261023_07_12_phillips_animado.html`: gráfico interactivo independiente.
- `resultados/20261023_07_12_phillips_estatico.png`: imagen estática del análisis.
- `resultados/20261023_07_12_datos_phillips.csv`: datos de cada burbuja.
- `resultados/20261023_07_12_cobertura.csv` y `20261023_07_12_meses_excluidos.csv`: auditoría de cobertura.

Para volver a trabajar, abre el mismo `.bat`. Ejecuta todas las celdas para descargar las cuatro fuentes, validarlas y recalcular el análisis. El comando `python scripts/05_ejecutar_todo.py` genera también el sitio y las tablas. Usa `--sin-actualizar-datos` solamente para reproducir un corte local. Una descarga fallida detiene la ejecución; no se sustituyen las cuatro fuentes por un conjunto parcial.

## Fuentes y referencias

- [INE — Ocupación y desocupación](https://www.ine.gob.cl/estadisticas-por-tema/mercado-laboral/ocupacion-y-desocupacion).
- [INE — IPC](https://www.ine.gob.cl/estadisticas-por-tema/precios-e-inflacion/indice-de-precios-al-consumidor).
- [INE — Remuneraciones y costos laborales](https://www.ine.gob.cl/estadisticas-por-tema/mercado-laboral/remuneraciones-y-costos-laborales).
- [Banco Central de Chile — IPoM junio de 2016, evidencia de la curva de Phillips](https://www.bcentral.cl/documents/33528/133297/bcch_archivo_164644_es.pdf/5c004bf3-b159-1ff4-78aa-b286738295b6).

Las URL exactas de los tabulados están en los extractores originales del proyecto.
